# 풍력발전량 예측 - 학습

`open/train/` 학습 데이터와 외부 예보로 모델을 학습해서 `model_artifacts/`에 저장한다.
평가 기간 데이터(`open/test/`)는 이 노트북에서 읽지 않는다.

| 저장 파일 | 내용 |
|---|---|
| `models_grp_kpx_group_{1,2,3}.pkl` | 그룹별 LightGBM (시드 10개) |
| `models_pool.pkl` | 3개 그룹 통합 LightGBM (시드 10개) |
| `calibrators.pkl` | isotonic 보정기, 구간별 오프셋 |
| `schema.pkl` | 피처 열 순서, 설정값 |

CPU에서 15~25분 정도 걸린다.

## 0. 설정

노트북은 `open/` 폴더가 있는 위치에서 실행한다.

In [1]:
import io
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

# 노트북이 있는 폴더(open/ 의 상위 폴더)를 기준 경로로 쓴다.
# 다른 위치에서 실행한다면 여기만 바꾸면 된다.
NB_ROOT = Path.cwd()
assert (NB_ROOT / "open").is_dir(), (
    f"open/ 을 찾을 수 없습니다: {NB_ROOT}\n"
    "노트북을 프로젝트 루트에서 실행하거나 NB_ROOT 를 직접 지정하세요.")

# cp949 콘솔에서 스크립트로 돌릴 때 출력 인코딩 오류 방지 (노트북에서는 건너뜀)
if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")

## 1. 피처 생성

LDAPS/GFS 격자 예보와 외부 예보로 시간별 피처를 만든다.
추론 노트북과 같은 코드이고, 노트북 하나만으로 돌아가도록 양쪽에 똑같이 넣어 두었다.

In [2]:
DATA_DIR = NB_ROOT / "open"

# 외부 예보 파일 선택. EXT_SUFFIX 가 있으면 external_{name}{EXT_SUFFIX}.csv 를 읽고,
# None 이면 COMPLIANT_EXT 값에 따라 _compliant 파일 또는 원본 파일을 읽는다.
COMPLIANT_EXT = True
EXT_SUFFIX = "_a1"

USE_MSM = True  # JMA MSM 지상풍 피처 (external_msm.csv)

def ext_csv(name):
    suf = EXT_SUFFIX if EXT_SUFFIX is not None else ("_compliant" if COMPLIANT_EXT else "")
    return DATA_DIR.parent / f"external_{name}{suf}.csv"

def day_block(t):
    # 발행 블록: D 01:00 ~ D+1 00:00 은 D-1 에 나온 같은 예보다.
    # lead, 중심 롤링처럼 뒤 시각을 보는 피처는 이 블록 안에서만 계산한다.
    # lag, diff 는 더 이전에 나온 값이라 제한하지 않는다.
    return (t - pd.Timedelta(hours=1)).dt.normalize()

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]

CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}

FARM_LAT, FARM_LON = 37.280, 128.963  # 터빈 위치 평균 (info.xlsx)

# (u 컬럼, v 컬럼, 풍속 컬럼 이름)
UV_PAIRS = {
    "ldaps": [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "ws10"),
        ("heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax", "ws50max"),
        ("heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin", "ws50min"),
        ("heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS", "ws5bl"),
    ],
    "gfs": [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "ws10"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "ws80"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "ws100"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "wspbl"),
        ("isobaricInhPa_850_u", "isobaricInhPa_850_v", "ws850"),
    ],
}

# 세제곱 피처를 추가할 풍속 (허브 높이 근처)
CUBED = {"ldaps": {"ws50max", "ws50min"}, "gfs": {"ws80", "ws100"}}

META_COLS = {"forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"}

# LDAPS 50MUmax/MVmax 는 성분별 최대값이라 벡터 크기 차이로 만든 spread 가 음수가 되기도 한다.
# True 면 u/v 변동폭으로 다시 계산한다 (피처 이름과 개수는 그대로). 최종 모델은 False.
SPREAD_FIX = False

def add_wind_speed(df, prefix):
    # 풍속은 격자별로 계산한 뒤 집계한다 (u/v 를 먼저 평균하면 방향이 상쇄돼 작아짐)
    for u_col, v_col, name in UV_PAIRS[prefix]:
        ws = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        df[name] = ws
        if name in CUBED[prefix]:
            df[f"{name}_cube"] = ws ** 3
    if SPREAD_FIX and prefix == "ldaps":
        du = df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]
        dv = df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]
        df["ws50osc"] = np.sqrt(du ** 2 + dv ** 2) / 2
    return df

# ldaps_test.csv 의 세 시각(2025-04-08 17시, 06-18 18시, 07-18 06시)은 16개 격자가 전부 비어 있다.
# 학습 데이터에는 결측이 없어서 모델이 이 NaN 을 0 처럼 다루게 되므로, 같은 발행 블록 안에서
# 시간 보간으로 채운다. 학습 데이터에는 영향이 없다.
IMPUTE_BLOCK_GAPS = True


def fill_block_gaps(df):
    """발행 블록 안에서 격자별로 시간 선형보간. 결측이 없으면 그대로 반환."""
    value_cols = [c for c in df.columns if c not in META_COLS]
    if not df[value_cols].isna().to_numpy().any():
        return df
    df = df.sort_values(["grid_id", "forecast_kst_dtm"])
    df[value_cols] = df.groupby(["grid_id", "data_available_kst_dtm"])[value_cols].transform(
        lambda s: s.interpolate(limit_direction="both"))
    return df.sort_index()


def prep(csv_path, prefix):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    if IMPUTE_BLOCK_GAPS:
        df = fill_block_gaps(df)
    return add_wind_speed(df, prefix)

def agg_mean(df, prefix, suffix="mean"):
    value_cols = [c for c in df.columns if c not in META_COLS]
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}_{suffix}" for c in agg.columns]
    return agg.reset_index()

def agg_std(df, prefix, cols):
    agg = df.groupby("forecast_kst_dtm")[cols].std()
    agg.columns = [f"{prefix}_{c}_std" for c in agg.columns]
    return agg.reset_index()

def nearest_grids(df, k):
    g = df[["grid_id", "latitude", "longitude"]].drop_duplicates("grid_id")
    d2 = (g["latitude"] - FARM_LAT) ** 2 + (g["longitude"] - FARM_LON) ** 2
    return g.loc[d2.nsmallest(k).index, "grid_id"].tolist()

def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out

def build_weather(ldaps_csv, gfs_csv):
    ldaps = prep(ldaps_csv, "ldaps")
    gfs = prep(gfs_csv, "gfs")

    # 전체 격자 평균 + 단지 인근 격자 평균(LDAPS 4개, GFS 1개) + 격자 간 표준편차
    base = agg_mean(ldaps, "ldaps").merge(agg_mean(gfs, "gfs"), on="forecast_kst_dtm", how="inner")
    near = agg_mean(ldaps[ldaps["grid_id"].isin(nearest_grids(ldaps, 4))], "ldaps", "n4").merge(
        agg_mean(gfs[gfs["grid_id"].isin(nearest_grids(gfs, 1))], "gfs", "n1"),
        on="forecast_kst_dtm", how="inner",
    )
    std = agg_std(ldaps, "ldaps", ["ws10", "ws50max", "ws5bl"]).merge(
        agg_std(gfs, "gfs", ["ws10", "ws80", "ws100"]), on="forecast_kst_dtm", how="inner"
    )
    w = base.merge(near, on="forecast_kst_dtm", how="left").merge(std, on="forecast_kst_dtm", how="left")

    # 시계열 피처 (lag/lead/diff/rolling). 뒤 시각을 보는 것은 발행 블록 안에서만.
    w = w.sort_values("forecast_kst_dtm").reset_index(drop=True)
    blk = day_block(w["forecast_kst_dtm"])
    for c in ["ldaps_ws50max_n4", "gfs_ws100_n1", "gfs_ws80_n1", "ldaps_ws50max_mean", "gfs_ws100_mean"]:
        s = w[c]
        for lag in (1, 2, 3):
            w[f"{c}_lag{lag}"] = s.shift(lag)
            w[f"{c}_lead{lag}"] = s.groupby(blk).shift(-lag)
        w[f"{c}_diff1"] = s - s.shift(1)
        w[f"{c}_roll6_mean"] = s.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).mean())
        w[f"{c}_roll6_std"] = s.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).std())
    for c in ["ldaps_ws50max_n4", "gfs_ws100_n1", "gfs_wspbl_n1", "ldaps_ws10_n4"]:
        s = w[c]
        for win in (12, 24):
            w[f"{c}_roll{win}_mean"] = s.groupby(blk).transform(
                lambda x: x.rolling(win, center=True, min_periods=1).mean())
            w[f"{c}_roll{win}_std"] = s.groupby(blk).transform(
                lambda x: x.rolling(win, center=True, min_periods=1).std())
        w[f"{c}_diff3"] = s - s.shift(3)
        w[f"{c}_lead6"] = s.groupby(blk).shift(-6)
        w[f"{c}_lag6"] = s.shift(6)

    # 50m 풍속 spread, 100m 풍향, 대기 안정도(850hPa - 2m 기온), 하루 풍속 요약
    if SPREAD_FIX:
        w["ws50_spread_n4"] = w["ldaps_ws50osc_n4"]
        w["ws50_spread_mean"] = w["ldaps_ws50osc_mean"]
        w = w.drop(columns=["ldaps_ws50osc_n4", "ldaps_ws50osc_mean"])
    else:
        w["ws50_spread_n4"] = w["ldaps_ws50max_n4"] - w["ldaps_ws50min_n4"]
        w["ws50_spread_mean"] = w["ldaps_ws50max_mean"] - w["ldaps_ws50min_mean"]
    ws100 = np.sqrt(w["gfs_heightAboveGround_100_100u_n1"] ** 2 + w["gfs_heightAboveGround_100_100v_n1"] ** 2)
    w["wd100_sin_n1"] = w["gfs_heightAboveGround_100_100u_n1"] / ws100.replace(0, np.nan)
    w["wd100_cos_n1"] = w["gfs_heightAboveGround_100_100v_n1"] / ws100.replace(0, np.nan)
    w["t_grad_n1"] = w["gfs_isobaricInhPa_850_t_n1"] - w["gfs_heightAboveGround_2_2t_n1"]
    day = (w["forecast_kst_dtm"] - pd.Timedelta(hours=1)).dt.date
    g = w.groupby(day)["gfs_ws100_n1"]
    w["ws100_day_mean"] = g.transform("mean")
    w["ws100_day_max"] = g.transform("max")
    w["ws100_day_min"] = g.transform("min")
    g2 = w.groupby(day)["ldaps_ws50max_n4"]
    w["ws50_day_mean"] = g2.transform("mean")
    w["ws50_day_max"] = g2.transform("max")
    return w

def add_icon_features(df):
    # DWD ICON 예보. 2022-11-24 이전은 NaN 이고 LightGBM 이 그대로 처리한다.
    ic = pd.read_csv(ext_csv("icon"), encoding="utf-8-sig", parse_dates=["time"])
    ic = ic.set_index("time")
    t = df["forecast_kst_dtm"]
    out = pd.DataFrame(index=df.index)
    ws100 = ic["wind_speed_100m"].reindex(t).to_numpy()
    wd = np.deg2rad(ic["wind_direction_100m"].reindex(t).to_numpy())
    out["icon_ws100"] = ws100
    out["icon_ws100_cube"] = ws100 ** 3
    out["icon_ws10"] = ic["wind_speed_10m"].reindex(t).to_numpy()
    out["icon_gust"] = ic["wind_gusts_10m"].reindex(t).to_numpy()
    out["icon_wd_sin"] = np.sin(wd)
    out["icon_wd_cos"] = np.cos(wd)
    out["icon_t2m"] = ic["temperature_2m"].reindex(t).to_numpy()
    out["icon_sp"] = ic["surface_pressure"].reindex(t).to_numpy()
    out["icon_minus_gfs_ws100"] = ws100 - df["gfs_ws100_n1"].to_numpy()
    out["icon_minus_ldaps_ws50"] = ws100 - df["ldaps_ws50max_n4"].to_numpy()
    s = pd.Series(ws100, index=df.index)
    blk = day_block(df["forecast_kst_dtm"])
    out["icon_ws100_roll6_mean"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).mean())
    out["icon_ws100_roll6_std"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).std())
    out["icon_ws100_diff1"] = s - s.shift(1)
    return pd.concat([df, out], axis=1)

def add_ext_features(df):
    # GEM(120m/80m), UKMO/JMA(10m) 예보와 모델 간 차이.
    # UKMO, JMA 는 2022년 초부터 있어서 ICON/GEM 이 없는 2022년 구간을 채워 준다.
    ic = pd.read_csv(ext_csv("icon"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    ge = pd.read_csv(ext_csv("gem"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    uk = pd.read_csv(ext_csv("ukmo"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    jm = pd.read_csv(ext_csv("jma"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    t = df["forecast_kst_dtm"]
    out = pd.DataFrame(index=df.index)

    ws120 = ge["wind_speed_120m"].reindex(t).to_numpy()
    gwd = np.deg2rad(ge["wind_direction_120m"].reindex(t).to_numpy())
    out["gem_ws120"] = ws120
    out["gem_ws120_cube"] = ws120 ** 3
    out["gem_ws80"] = ge["wind_speed_80m"].reindex(t).to_numpy()
    out["gem_gust"] = ge["wind_gusts_10m"].reindex(t).to_numpy()
    out["gem_wd_sin"] = np.sin(gwd)
    out["gem_wd_cos"] = np.cos(gwd)
    out["gem_minus_icon"] = ws120 - ic["wind_speed_100m"].reindex(t).to_numpy()
    out["gem_minus_gfs"] = ws120 - df["gfs_ws100_n1"].to_numpy()
    s = pd.Series(ws120, index=df.index)
    blk = day_block(df["forecast_kst_dtm"])
    out["gem_ws120_roll6_mean"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).mean())
    out["gem_ws120_roll6_std"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).std())
    out["gem_ws120_diff1"] = s - s.shift(1)

    icon10 = ic["wind_speed_10m"].reindex(t).to_numpy()
    uws = uk["wind_speed_10m"].reindex(t).to_numpy()
    jws = jm["wind_speed_10m"].reindex(t).to_numpy()
    uwd = np.deg2rad(uk["wind_direction_10m"].reindex(t).to_numpy())
    jwd = np.deg2rad(jm["wind_direction_10m"].reindex(t).to_numpy())
    out["ukmo_ws10"] = uws
    out["ukmo_gust"] = uk["wind_gusts_10m"].reindex(t).to_numpy()
    out["jma_ws10"] = jws
    out["ukmo_wd_sin"] = np.sin(uwd)
    out["ukmo_wd_cos"] = np.cos(uwd)
    out["jma_wd_sin"] = np.sin(jwd)
    out["jma_wd_cos"] = np.cos(jwd)
    out["ukmo_minus_jma"] = uws - jws
    out["ukmo_minus_icon10"] = uws - icon10
    out["jma_minus_icon10"] = jws - icon10

    if USE_MSM:
        # JMA MSM 은 전날 00UTC 실행분만 모아 둔 자료라 _a1 전처리 없이 바로 읽는다
        ms = pd.read_csv(DATA_DIR.parent / "external_msm.csv", encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
        ms = ms[~ms.index.duplicated(keep="last")].sort_index()
        mws = ms["msm_ws10"].reindex(t).to_numpy()
        mwd = np.deg2rad(ms["msm_wd10"].reindex(t).to_numpy())
        out["msm_ws10"] = mws
        out["msm_wd_sin"] = np.sin(mwd)
        out["msm_wd_cos"] = np.cos(mwd)
        out["msm_minus_jma"] = mws - jws
        out["msm_minus_icon10"] = mws - icon10
        sm = pd.Series(mws, index=df.index)
        out["msm_ws10_roll6_mean"] = sm.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).mean())
        out["msm_ws10_roll6_std"] = sm.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).std())
    return pd.concat([df, out], axis=1)

## 2. 학습

터빈 정지/고장 시간을 빼고 그룹별 모델과 통합 모델을 학습한 뒤, OOF 예측으로 isotonic 보정기와 구간별 오프셋을 만든다.
정지/고장 판단에는 학습 기간 SCADA만 쓴다.

In [3]:
USE_NACELLE = True   # 나셀 풍속 기준 고장 시간도 학습에서 제외

SEEDS = (42, 7, 123, 2024, 777, 1, 2, 3, 4, 5)
OOF_SEEDS = (42, 7, 123)

def lgbm(seed):
    return LGBMRegressor(
        objective="regression_l1", n_estimators=1000, learning_rate=0.05,
        num_leaves=42, min_child_samples=84, feature_fraction=0.7852700067948886,
        bagging_fraction=0.7438158035038887, lambda_l1=0.35402126361409586,
        lambda_l2=6.3435928493560105,
        bagging_freq=1, random_state=seed, n_jobs=-1, verbose=-1,
    )

POOLED_ALPHA = 0.7  # 통합 모델 블렌드 비중

# 그룹별 (SCADA 파일, 터빈 목록, 10분 정격 에너지 kWh)
SCADA_FILES = {
    "kpx_group_1": ("scada_vestas_train.csv", [f"vestas_wtg{i:02d}" for i in range(1, 7)], 600),
    "kpx_group_2": ("scada_vestas_train.csv", [f"vestas_wtg{i:02d}" for i in range(7, 13)], 600),
    "kpx_group_3": ("scada_unison_train.csv", [f"unison_wtg{i:02d}" for i in range(1, 6)], 700),
}

def outage_mask(times):
    # 그룹은 발전 중(중앙값 > 정격 15%)인데 특정 터빈만 거의 0인 시간.
    # 1기만 멈춰도 그룹 출력이 17% 정도 떨어지고 기상으로는 설명이 안 되므로 학습에서 뺀다.
    out = {}
    for g, (fname, turbs, rated10) in SCADA_FILES.items():
        s = pd.read_csv(DATA_DIR / "train" / fname, encoding="utf-8-sig")
        s["kst_dtm"] = pd.to_datetime(s["kst_dtm"])
        p = s.set_index("kst_dtm")[[f"{t}_power_kw10m" for t in turbs]]
        p = p.clip(upper=rated10 * 1.3)  # VESTAS 센서 이상값 제거
        hourly = p.resample("h", label="right", closed="right").mean()
        med = hourly.median(axis=1)
        active = med > 0.15 * rated10
        cnt = hourly.lt(np.maximum(0.05 * med, 2), axis=0).sum(axis=1).where(active, 0)
        out[g] = (cnt.reindex(times).fillna(0) >= 1).to_numpy()
    return out

def nacelle_mask(times):
    # outage_mask 가 못 잡는 경우: 나셀 풍속은 4 m/s 이상인데 출력이
    # 파워커브(풍속 구간별 출력 중앙값) 기대치의 35% 미만인 시간.
    out = {}
    for g, (fname, turbs, rated10) in SCADA_FILES.items():
        s = pd.read_csv(DATA_DIR / "train" / fname, encoding="utf-8-sig")
        s["kst_dtm"] = pd.to_datetime(s["kst_dtm"])
        s = s.set_index("kst_dtm")
        bins = np.arange(0, 26, 1.0)
        ws_all, pw_all = [], []
        for t in turbs:
            pair = s[[f"{t}_ws", f"{t}_power_kw10m"]].dropna()
            ws_all.append(pair[f"{t}_ws"].values)
            pw_all.append(pair[f"{t}_power_kw10m"].clip(0, rated10 * 1.3).values)
        ws_all, pw_all = np.concatenate(ws_all), np.concatenate(pw_all)
        idx = np.digitize(ws_all, bins)
        curve = {b: np.median(pw_all[idx == b]) for b in np.unique(idx) if (idx == b).sum() > 50}
        cnt = pd.Series(0.0, index=s.index)
        for t in turbs:
            w = s[f"{t}_ws"]
            p = s[f"{t}_power_kw10m"].clip(0, rated10 * 1.3)
            exp = w.map(lambda x: curve.get(int(np.digitize(x, bins)), np.nan) if pd.notna(x) else np.nan)
            fault = ((w >= 4) & (exp > 0.05 * rated10) & (p < 0.35 * exp)).fillna(False)
            cnt = cnt + fault.astype(float)
        hourly = cnt.resample("h", label="right", closed="right").mean()
        out[g] = (hourly.reindex(times).fillna(0) >= 1).to_numpy()
    return out

def fit_bin_shifts(p, y, cap, n_bins=8, grid=None, min_rows=400, min_elig=50, shrink=0.5):
    # OOF 예측을 분위 구간으로 나누고, 구간마다 대회 점수(NMAE 0.5 + FiCR 0.5)가
    # 가장 좋아지는 오프셋을 찾는다. L1 은 NMAE 기준이라 FiCR 의 6%/8% 구간 보상에는 딱 맞지 않는다.
    # 범위를 넓히거나(±4%) 구간을 늘리면 홀드아웃은 좋아졌지만 LB 에서는 떨어져서
    # ±2.5%cap, 8구간, 절반만 반영(shrink=0.5)으로 보수적으로 잡았다.
    if grid is None:
        grid = np.arange(-0.025, 0.0251, 0.005)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    bin_id = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    elig = y >= 0.1 * cap
    n_elig, y_elig_sum = max(elig.sum(), 1), y[elig].sum()
    deltas = np.zeros(len(edges) - 1)
    for b in range(len(edges) - 1):
        sel = bin_id == b
        sel_e = sel & elig
        if sel.sum() < min_rows or sel_e.sum() < min_elig:
            continue
        ys, ps = y[sel_e], p[sel_e]
        best_d, best_v = 0.0, None
        for d in grid * cap:
            err = np.abs(ys - (ps + d)) / cap
            won = np.where(err <= 0.06, 4.0, np.where(err <= 0.08, 3.0, 0.0))
            v = -0.5 * err.sum() / n_elig + 0.5 * (won * ys).sum() / (4.0 * y_elig_sum)
            if best_v is None or v > best_v:
                best_v, best_d = v, d
        deltas[b] = shrink * best_d
    return edges, deltas


ART = NB_ROOT / "model_artifacts"


def build_train_matrix():
    """학습 피처 행렬 X_train 과, 라벨과 시각이 들어 있는 train_df 를 만든다."""
    labels = pd.read_csv(DATA_DIR / "train" / "train_labels.csv", encoding="utf-8-sig")
    labels["kst_dtm"] = pd.to_datetime(labels["kst_dtm"])
    weather = build_weather(DATA_DIR / "train" / "ldaps_train.csv",
                               DATA_DIR / "train" / "gfs_train.csv")
    train_df = labels.rename(columns={"kst_dtm": "forecast_kst_dtm"}).merge(
        weather, on="forecast_kst_dtm", how="left")
    train_df = add_ext_features(add_icon_features(train_df))
    X_train = pd.concat(
        [calendar_features(train_df["forecast_kst_dtm"]),
         train_df.drop(columns=["forecast_kst_dtm", *TARGET_COLS])],
        axis=1,
    )
    return train_df, X_train


def build_exclusion(train_df):
    """학습에서 뺄 시간 (터빈 정지 + 나셀 기준 고장)."""
    times = pd.DatetimeIndex(train_df["forecast_kst_dtm"])
    excl = outage_mask(times)
    if USE_NACELLE:
        nf = nacelle_mask(times)
        print(f"  나셀-고장 추가제외: {sum(int((nf[g] & ~excl[g]).sum()) for g in nf):,}h")
        excl = {g: excl[g] | nf[g] for g in excl}
    for g in TARGET_COLS:
        print(f"  {g}: 제외 {int(excl[g].sum()):,}h")
    return excl


def fit_pooled(X_train, train_df, excl):
    """3개 그룹 데이터를 세로로 쌓고 group_id 를 붙여 하나의 모델로 학습한다.

    반환값: (시드별 모델 리스트, 학습 열 순서, 그룹별 OOF 예측)
    """
    frames, ys, times, gids = [], [], [], []
    for gi, g in enumerate(TARGET_COLS):
        m = train_df[g].notna().to_numpy() & ~excl[g]
        f = X_train.loc[m].copy()
        f["group_id"] = gi
        frames.append(f)
        ys.append((train_df.loc[m, g] / CAPACITY_KWH[g]).to_numpy())
        times.append(train_df.loc[m, "forecast_kst_dtm"].to_numpy())
        gids.append(np.full(m.sum(), gi))
    P = pd.concat(frames, ignore_index=True)
    y = np.concatenate(ys)
    t = np.concatenate(times)
    gid = np.concatenate(gids)
    w = y.copy()

    models = [lgbm(s).fit(P, y, sample_weight=w) for s in SEEDS]

    # OOF: 같은 시각의 3개 그룹 행은 같은 폴드에 넣는다
    uniq = np.unique(t)
    fold_of = dict(zip(uniq, np.random.RandomState(0).randint(0, 5, len(uniq))))
    fold_ids = np.array([fold_of[v] for v in t])
    oof = np.zeros(len(P))
    for k in range(5):
        fit_i, prd_i = fold_ids != k, fold_ids == k
        fp = [lgbm(s).fit(P.loc[fit_i], y[fit_i], sample_weight=w[fit_i]).predict(P.loc[prd_i])
              for s in OOF_SEEDS]
        oof[prd_i] = np.mean(fp, axis=0)
    oof_pred = {g: oof[gid == gi] * CAPACITY_KWH[g] for gi, g in enumerate(TARGET_COLS)}
    return models, list(P.columns), oof_pred


def main():
    t0 = time.time()
    ART.mkdir(exist_ok=True)
    print("=" * 70)
    print("학습 코드 (train) — 평가 기간 데이터 미사용")
    print("=" * 70)

    print("\n[1/5] 학습 피처 구축...")
    train_df, X_train = build_train_matrix()
    print(f"  X_train: {X_train.shape}")

    print("\n[2/5] 오염 라벨 제외 마스크...")
    excl = build_exclusion(train_df)

    print("\n[3/5] 통합(pooled) 모델 학습 + OOF...")
    pool_models, pool_columns, pool_oof = fit_pooled(X_train, train_df, excl)
    with open(ART / "models_pool.pkl", "wb") as f:
        pickle.dump(pool_models, f)
    print(f"  저장: models_pool.pkl ({len(pool_models)} 시드)")

    print("\n[4/5] 그룹별 모델 학습 + OOF + 보정층 적합...")
    calib = {}
    for g in TARGET_COLS:
        cap = CAPACITY_KWH[g]
        mask = train_df[g].notna() & ~excl[g]
        y_fit = train_df.loc[mask, g]
        sw = (y_fit / cap).to_numpy()          # FiCR 이 발전량 가중이라 고출력 시간에 가중치
        Xf = X_train.loc[mask]

        models = [lgbm(s).fit(Xf, y_fit, sample_weight=sw) for s in SEEDS]
        with open(ART / f"models_grp_{g}.pkl", "wb") as f:
            pickle.dump(models, f)

        # 5-fold OOF (3시드 평균)를 통합 모델 OOF 와 같은 비율로 섞는다
        oof = np.zeros(len(Xf))
        for tr, te in KFold(5, shuffle=True, random_state=0).split(np.arange(len(Xf))):
            fold_preds = [lgbm(s).fit(Xf.iloc[tr], y_fit.iloc[tr], sample_weight=sw[tr])
                          .predict(Xf.iloc[te]) for s in OOF_SEEDS]
            oof[te] = np.mean(fold_preds, axis=0)
        oof = (1 - POOLED_ALPHA) * oof + POOLED_ALPHA * pool_oof[g]

        # isotonic 보정 (발전량 가중)
        iso = IsotonicRegression(y_min=0, y_max=cap, out_of_bounds="clip")
        iso.fit(np.clip(oof, 0, cap), y_fit, sample_weight=sw)

        # 보정된 OOF 로 구간별 오프셋 계산
        oof_cal = iso.predict(np.clip(oof, 0, cap))
        edges, deltas = fit_bin_shifts(oof_cal, y_fit.to_numpy(), cap)

        calib[g] = dict(iso=iso, edges=edges, deltas=deltas, cap=cap,
                        n_train=int(mask.sum()))
        print(f"  {g}: 학습행 {int(mask.sum()):,}  시프트 {np.round(deltas / cap * 100, 2)}%cap")

    with open(ART / "calibrators.pkl", "wb") as f:
        pickle.dump(calib, f)

    print("\n[5/5] 피처 스키마 저장...")
    schema = dict(
        X_columns=list(X_train.columns),
        pool_columns=pool_columns,
        target_cols=list(TARGET_COLS),
        capacity=dict(CAPACITY_KWH),
        pooled_alpha=POOLED_ALPHA,
        seeds=list(SEEDS),
        oof_seeds=list(OOF_SEEDS),
        config=dict(EXT_SUFFIX=EXT_SUFFIX, USE_MSM=USE_MSM, USE_NACELLE=USE_NACELLE),
    )
    with open(ART / "schema.pkl", "wb") as f:
        pickle.dump(schema, f)
    print(f"  피처 {len(schema['X_columns'])}열 · 통합모델 {len(pool_columns)}열")

    tot = sum(p.stat().st_size for p in ART.glob("*.pkl"))
    print(f"\n완료: {ART.name}/  ({tot / 1e6:.1f} MB, {time.time() - t0:.0f}s)")
    print("다음 단계: submit_inference.ipynb 실행")

## 3. 실행

In [4]:
main()

학습 코드 (train) — 평가 기간 데이터 미사용

[1/5] 학습 피처 구축...
  X_train: (26304, 295)

[2/5] 오염 라벨 제외 마스크...
  나셀-고장 추가제외: 3,640h
  kpx_group_1: 제외 4,568h
  kpx_group_2: 제외 3,008h
  kpx_group_3: 제외 3,337h

[3/5] 통합(pooled) 모델 학습 + OOF...
  저장: models_pool.pkl (10 시드)

[4/5] 그룹별 모델 학습 + OOF + 보정층 적합...
  kpx_group_1: 학습행 21,634  시프트 [ 0.    1.25  1.25 -0.75 -1.25  0.    0.    0.5 ]%cap
  kpx_group_2: 학습행 23,195  시프트 [ 0.    1.25  1.25 -1.25 -1.    0.75  1.    0.25]%cap
  kpx_group_3: 학습행 14,201  시프트 [ 0.    0.    1.25  1.   -0.25 -1.25  0.25  1.25]%cap

[5/5] 피처 스키마 저장...
  피처 295열 · 통합모델 296열

완료: model_artifacts/  (187.2 MB, 855s)
다음 단계: submit_inference.ipynb 실행
